# 02 Dimensionality Reduction
PCA, t-SNE, and UMAP are applied to the ESM2 protein embeddings to visualise family structure in 2D. Clustering quality is evaluated using the Adjusted Rand Index across all methods and configurations.

## Data Loading
Annotations and embeddings are loaded from disk.

In [ ]:
import pathlib, sys, os

project_root = pathlib.Path.cwd()
for _ in range(5):
    if (project_root / "environment.yml").exists():
        break
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.data_loader import load_data
from src import reduction, evaluation

df2, emb2 = load_data()

## PCA
Embeddings are reduced to 2D with PCA to give a first look at family structure. Proteins are coloured by family; only families with at least 2 members are shown.

In [ ]:
embeddings_pca, _ = reduction.pca(emb2, n_components=2)
print(embeddings_pca.shape)

In [ ]:
family_counts = df2["family"].value_counts()
common_families = family_counts[family_counts >= 2].index
mask = df2["family"].isin(common_families)
print("Proteins in families with ≥2 members:", mask.sum())
print("Number of such families:", df2[mask]["family"].unique().shape[0])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import pandas as pd

family_labels = df2["family"][mask]
codes, uniques = pd.factorize(family_labels)
n = len(uniques)
color_list = (
    list(plt.cm.tab10.colors) +
    list(plt.cm.tab20b.colors[::2]) +
    list(plt.cm.tab20c.colors[::2])
)[:n]
cmap = mcolors.ListedColormap(color_list)

top8 = family_counts.head(8).index.tolist()

def make_legend_handles(uniques, color_list, top8):
    handles = []
    has_other = False
    for i, fam in enumerate(uniques):
        if fam in top8:
            handles.append(mpatches.Patch(color=color_list[i], label=fam))
        else:
            has_other = True
    if has_other:
        handles.append(mpatches.Patch(color="lightgray", label="Other"))
    return handles

legend_handles = make_legend_handles(uniques, color_list, top8)

plt.figure(figsize=(12, 6))
scatter = plt.scatter(embeddings_pca[mask, 0], embeddings_pca[mask, 1], c=codes, cmap=cmap)
plt.xlabel("First Principal Component")
plt.ylabel("Second Principal Component")
plt.title("PCA of Protein Embeddings")
plt.legend(handles=legend_handles, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
right_cluster = df2[mask][embeddings_pca[mask, 0] > 1]
print(right_cluster["family"].value_counts())


In [ ]:
bottom_cluster = df2[mask][embeddings_pca[mask, 1] < -0.6]
print(bottom_cluster["family"].value_counts())

### Cumulative Explained Variance
Full PCA is fitted to find how many dimensions are needed to capture 90% and 95% of the variance, informing the choice of reduced dimensionality for downstream clustering.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

_, pca_full = reduction.pca(emb2, n_components=emb2.shape[1])

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

n_90 = np.argmax(cumvar >= 0.90) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1

print(f"Dimensions to explain 90% variance: {n_90}")
print(f"Dimensions to explain 95% variance: {n_95}")
print(f"Total dimensions: {emb2.shape[1]}")

plt.figure(figsize=(10, 5))
plt.plot(cumvar)
plt.axhline(y=0.90, color="r", linestyle="--", label="90% variance")
plt.axhline(y=0.95, color="g", linestyle="--", label="95% variance")
plt.axvline(x=n_90, color="r", linestyle=":", alpha=0.7)
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA cumulative explained variance")
plt.legend()
plt.savefig(project_root / "outputs" / "figures" / "pca_variance.png")
plt.show()

## t-SNE
t-SNE is run at three perplexity values (10, 30, 50) to assess sensitivity. The perplexity=30 projection is shown; all three are retained for ARI comparison below.

In [ ]:
embeddings_tsne_p10 = reduction.tsne(emb2, perplexity=10)
embeddings_tsne_p50 = reduction.tsne(emb2, perplexity=50)
embeddings_tsne_p30 = reduction.tsne(emb2, perplexity=30)

plt.figure(figsize=(12, 6))
plt.scatter(embeddings_tsne_p30[mask, 0], embeddings_tsne_p30[mask, 1], c=codes, cmap=cmap)
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.title("t-SNE of Protein Embeddings (Perplexity=30)")
plt.legend(handles=legend_handles, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## UMAP
UMAP is run at three `n_neighbors` values (4, 10, 15) to assess sensitivity. The n_neighbors=4 projection is used in the final comparison figure.

In [ ]:
embeddings_umap_n4  = reduction.umap(emb2, n_neighbors=4)
embeddings_umap_n10 = reduction.umap(emb2, n_neighbors=10)
embeddings_umap_n15 = reduction.umap(emb2, n_neighbors=15)

plt.figure(figsize=(12, 6))
plt.scatter(embeddings_umap_n15[mask, 0], embeddings_umap_n15[mask, 1], c=codes, cmap=cmap)
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.title("UMAP of Protein Embeddings (n_neighbors=15)")
plt.legend(handles=legend_handles, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Clustering Quality (ARI)
K-means (k=29, matching the number of families) is applied to each reduced representation. The Adjusted Rand Index measures how well the clusters align with known protein families.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA
import pandas as pd

# ground truth labels — masked proteins only
family_counts = df2["family"].value_counts()
common_families = family_counts[family_counts >= 2].index
mask = df2["family"].isin(common_families)
true_labels, _ = pd.factorize(df2[mask]["family"])

# PCA 50D on masked proteins
pca50 = PCA(n_components=50)
emb_pca50_masked = pca50.fit_transform(emb2[mask])

# all embedding spaces — masked proteins only
spaces = {
    "PCA 2D":    embeddings_pca[mask],
    "PCA 50D":   emb_pca50_masked,
    "Full 320D": emb2[mask],
    "t-SNE (p=10)": embeddings_tsne_p10[mask],
    "t-SNE (p=30)": embeddings_tsne_p30[mask],
    "t-SNE (p=50)": embeddings_tsne_p50[mask],
    "UMAP (n=3)":   embeddings_umap_n15[mask],
    "UMAP (n=4)":   embeddings_umap_n4[mask],
    "UMAP (n=10)":  embeddings_umap_n10[mask],
}

print(f"{'Method':<20} {'ARI':>6}")
print("-" * 28)
for name, emb in spaces.items():
    labels = KMeans(n_clusters=29, random_state=42).fit_predict(emb)
    ari = adjusted_rand_score(true_labels, labels)
    print(f"{name:<20} {ari:.3f}")

In [ ]:
import numpy as np

np.save(project_root / "data" / "processed" / "umap_2d.npy", embeddings_umap_n4)
np.save(project_root / "data" / "processed" / "pca_2d.npy", embeddings_pca)
np.save(project_root / "data" / "processed" / "tsne_2d.npy", embeddings_tsne_p30)

## Comparison Figure
PCA, t-SNE (perplexity=30), and UMAP (n_neighbors=4) are plotted side by side for a direct visual comparison of how well each method separates protein families.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scatter_data = [
    (embeddings_pca, "PCA", "PC1", "PC2"),
    (embeddings_tsne_p30, "t-SNE (perplexity=30)", "t-SNE 1", "t-SNE 2"),
    (embeddings_umap_n4, "UMAP (n_neighbors=4)", "UMAP 1", "UMAP 2"),
]

for ax, (emb, title, xlabel, ylabel) in zip(axes, scatter_data):
    ax.scatter(emb[mask, 0], emb[mask, 1], c=codes, cmap=cmap, s=10, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

axes[-1].legend(handles=legend_handles, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)
plt.suptitle("Dimensionality reduction comparison protein embeddings coloured by family", y=1.02)
plt.tight_layout()

out = project_root / "outputs" / "figures" / "dimred_comparison.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved to {out}")
plt.show()

## Summary
- t-SNE (perplexity=30) achieves the highest ARI (0.675), best separating protein families in 2D
- UMAP (n_neighbors=4) is competitive (ARI 0.633) and faster than t-SNE
- PCA 50D (ARI 0.405) outperforms the full 320D embeddings (ARI 0.381) for clustering, suggesting PCA removes noise
- PCA 2D (ARI 0.166) loses too much structure for clustering but is useful for quick visual inspection
- 90% of embedding variance is captured in far fewer than 320 dimensions, confirming redundancy in the raw ESM2 space